# Data Preparation & Preprocessing

Dieses Notebook bereitet den Rohdatensatz für die Modellierung vor.
Die Pipeline folgt dem **dreistufigen Cleaning-Schema** aus `01_exploration.ipynb`:

```
Raw Data  (~94 Mio. Zeilen)
    │
    ▼
[Phase 1]  Strukturelles Cleaning  (Vor Split)
           Domainregeln · kein Leakage-Risiko
           → data/interim/zh-tram-structural-clean.parquet
    │
    ▼
[Phase 2]  Train / Test Split  —  temporal, kein Shuffle
           2023–2024 = Train  ·  2025 = Test
           → data/interim/train_raw.parquet
           → data/interim/test_raw.parquet
    │
    ▼
[Phase 3]  Statistisches Cleaning  (Post-Split)
           Parameter nur auf Train berechnet → auf Test anwenden
           → data/processed/train_prepared.parquet
           → data/processed/test_prepared.parquet
    │
    ▼
[Phase 4]  Feature Engineering
           Zeitfeatures · Extremwert-Flags · Encodings
           → data/processed/train_features.parquet
           → data/processed/test_features.parquet
```

**Leakage-Prinzip:**  
Was statistische Parameter aus den Daten lernt (Rolling Mean, IQR-Grenzen, Encodings)  
muss **nach dem Split** auf Trainingsdaten gefittet und dann auf Testdaten angewendet werden.

## Setup

### Imports

In [ ]:
import polars as pl
from pathlib import Path

from wgnd.core.theme import setup
from wgnd.core._output import section_header, log, success, warn

from zh_tram_flow.config import PATHS, RANDOM_SEED
from zh_tram_flow.cleaning import (
    structural_cleaning_pipeline,
    impute_meteo_rolling,
    report_step,
    METEO_COLS,
)

setup()

%load_ext autoreload
%autoreload 2

### Pfade & Datensatz

Alle Pfade kommen aus `zh_tram_flow.config.PATHS` — zentral in `src/zh_tram_flow/config.py`.  
Der Rohdatensatz ist `zh-tram-data-master.parquet` aus Phase 0 (sf_data-research), 24 Spalten, ~94 Mio. Zeilen.

In [ ]:
DATA      = PATHS["raw"]       / "zh-tram-data-master.parquet"
INTERIM   = PATHS["interim"]
PROCESSED = PATHS["processed"]

INTERIM.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)

# Lazy Scan — kein RAM-Verbrauch bis collect()
lf_raw = pl.scan_parquet(DATA)

n_raw    = lf_raw.select(pl.len()).collect().item()
date_min = lf_raw.select(pl.col("operating_date").min()).collect().item()
date_max = lf_raw.select(pl.col("operating_date").max()).collect().item()

log(f"Datensatz: {n_raw:,} Zeilen · {len(lf_raw.collect_schema())} Spalten")
log(f"Zeitraum:  {date_min} bis {date_max}")

## EDA-Findings — Cleaning-Agenda

Aus `01_exploration.ipynb` — diese Tabelle bestimmt was in welcher Phase bereinigt wird.

| Topic | Befund | Empfehlung | Phase |
|:---|:---|:---|:---:|
| **Delay** | Extreme Werte ±8.3h — physikalisch nicht plausibel | Rausfiltern `\|delay\| > 3.600s` | 1 |
| **Delay** | 74.669 Zeilen: Schedule vorhanden, aber kein Delay | Herausfiltern | 1 |
| **BPUIC** | 0.10% anomale IDs > 100.000.000 | Rausfiltern | 1 |
| **Meteo** | `humidity` > 100% — Sensor-Kalibrierungsdrift | `.clip(0, 100)` | 1 |
| **Datenqualität** | 1.72% Duplikate | `distinct()` | 1 |
| **Events** | 78.5% null — Normalfall kein Event | `null` → `"kein_event"` | 1 |
| **District** | 6.87% null — Haltestellen außerhalb Stadtgebiet | `null` → `"ausserhalb"` | 1 |
| **Meteo** | Stündliche Messausfälle (~0.14–0.35%), zeitlich klumpend | Forward/Backward Fill ±2h | 3 |
| **Meteo** | `precipitation` zero-inflated | Flag `hat_regen` + Wert | 4 |
| **Delay** | Skewness 38–43 — non-linear, long tail | Keine Bereinigung — XGBoost | — |
| **Meteo** | Wetter→Delay: r max 0.03 linear | Schwellenwert-Flags als Features | 4 |

> **Prognose:** ~1,7 Mio. Zeilen (~2%) werden durch Phase 1 entfernt.  
> Nach Phase 1 verbleiben **~86–87 Mio. Zeilen**.

In [ ]:
## Phase 1 — Strukturelles Cleaning (Vor Split)

Strukturelles Cleaning entfernt klar falsche Werte auf Basis von Domainwissen —  
ohne statistische Parameter zu berechnen. **Kein Leakage-Risiko.**

**Pipeline aus `zh_tram_flow.cleaning.structural_cleaning_pipeline`:**

| Schritt | Funktion | Begründung | Erwartete Reduktion |
|:---|:---|:---|:---|
| 1 | `remove_duplicates` | 1.72% doppelte Zeilen (GTFS-Join-Artefakt) | ~1,5 Mio. |
| 2 | `filter_bpuic_anomalies` | 9-stellige IDs außerhalb VBZ-Bereich | ~91k |
| 3 | `filter_delay_mismatch` | Schedule ok, Delay null (Übertragungsfehler) | ~75k |
| 4 | `filter_extreme_delays` | `\|delay\| > 3.600s` — physikalisch nicht plausibel | ~5k |
| 5 | `clip_physical_bounds` | `humidity` > 100% auf 100 kappen | keine Zeilen entfernt |
| 6 | `fill_category_nulls` | `district_name/event_*` Nulls → Kategorie-Labels | keine Zeilen entfernt |

section_header('Phase 1 — Strukturelles Cleaning')

n_before = lf_raw.select(pl.len()).collect().item()
log(f"Vor Cleaning: {n_before:,} Zeilen")
print()

# Lazy Pipeline aus src/zh_tram_flow/cleaning.py — collect() einmalig am Ende
lf_clean = structural_cleaning_pipeline(lf_raw)
df_clean = lf_clean.collect()
n_after  = len(df_clean)

print()
report_step("Gesamt Phase 1", n_before, n_after)
print()
success(f"Strukturelles Cleaning abgeschlossen: {n_after:,} Zeilen verbleiben")

In [ ]:
section_header('Phase 1 — Export')

out = INTERIM / "zh-tram-structural-clean.parquet"
df_clean.write_parquet(out)

log(f"Exportiert: {out}")
log(f"Shape:      {df_clean.shape}")

## Phase 2 — Train / Test Split

### Split-Strategie: Temporal Split (kein Random Shuffle)

**Warum temporal?**  
Bei Zeitreihendaten würde ein zufälliger Split Datenleck erzeugen: das Modell würde auf Daten
trainieren die zeitlich nach den Testdaten liegen — und damit zukünftige Wettermuster, Saisonalitäten
und Linien-Veränderungen "kennen". Ein temporaler Split verhindert das vollständig.

**Aufteilung:**
- **Train:** 2023 + 2024 — ~2 Jahre, Schätzung ~68–72 Mio. Zeilen nach Phase 1  
- **Test:** 2025 — ~1 Jahr, Schätzung ~15–18 Mio. Zeilen

**Sampling für das Modelltraining?**  
XGBoost kann mit 65+ Mio. Zeilen arbeiten, aber Trainingszeit steigt linear.  
Optionen:

| Ansatz | Vorteil | Nachteil |
|:---|:---|:---|
| Vollständiges Training | Maximale Genauigkeit | Stunden-Trainingszeit |
| Stratifiziertes Sampling (line_name × month) | Schneller, repräsentativ | Weniger seltene Kombinationen |

→ **Entscheidung vor Phase 4.** Die Splits hier enthalten alle Daten — Sampling kann im  
Modellierungs-Notebook optional angewendet werden.

**Kein Re-Fitting auf Test:**  
Phase 3 (Meteo Forward-Fill) wird auf Train und Test mit derselben Funktion angewendet —  
keine Parameter werden gelernt. Zukünftige statistische Imputation: erst auf Train fitten.

In [ ]:
section_header('Phase 2 — Train / Test Split')

# Temporal Split: 2025 = Test, 2023–2024 = Train
df_train = df_clean.filter(pl.col("operating_date").dt.year() < 2025)
df_test  = df_clean.filter(pl.col("operating_date").dt.year() == 2025)

log(f"Train: {len(df_train):,} Zeilen  ({len(df_train)/len(df_clean)*100:.1f}%)")
log(f"Test:  {len(df_test):,} Zeilen  ({len(df_test)/len(df_clean)*100:.1f}%)")

print()
log("Jahresverteilung Train:")
display(
    df_train
    .select(pl.col("operating_date").dt.year().alias("jahr"))
    .group_by("jahr").agg(pl.len().alias("n_fahrten"))
    .sort("jahr")
)

log("Jahresverteilung Test:")
display(
    df_test
    .select(pl.col("operating_date").dt.year().alias("jahr"))
    .group_by("jahr").agg(pl.len().alias("n_fahrten"))
    .sort("jahr")
)

# Export
df_train.write_parquet(INTERIM / "train_raw.parquet")
df_test.write_parquet(INTERIM  / "test_raw.parquet")

print()
success(f"Split exportiert → {INTERIM}")
log(f"  train_raw.parquet  ({len(df_train):,} Zeilen)")
log(f"  test_raw.parquet   ({len(df_test):,} Zeilen)")

## Phase 3 — Statistisches Cleaning (Post-Split)

Diese Schritte **müssen nach dem Split** erfolgen, da sie Parameter aus den Daten berechnen könnten.  
Hier angewendet: Meteo Forward/Backward Fill — technisch ohne gelernte Parameter, aber strukturell
an dieser Stelle richtig damit zukünftige Erweiterungen (z.B. stations-spezifische Median-Imputation)
kein Leakage erzeugen.

**Schritte:**

1. **Meteo Forward/Backward Fill** — füllt stündliche Messausfälle (~0.14–0.35%)  
   Funktioniert über `impute_meteo_rolling()` aus `zh_tram_flow.cleaning`.  
   Sortiert nach `arrival_schedule` → propagiert letzten gültigen Stundenwert über Ausfallstunden.  
   `flood_intensity` separat: `fill_null(0)` (Ereignis-Indikator, kein Mittelwert sinnvoll).

2. **IQR-Check nach Phase 1** *(optional)* — Validierung dass Delay-Bereinigung korrekt war.

> Gleiche Funktion auf Train und Test — kein separates "Fitting" notwendig für diesen Schritt.

In [ ]:
section_header('Phase 3 — Meteo-Imputation (Forward/Backward Fill)')

log("Train — Nulls vor Imputation:")
for col in METEO_COLS + ["flood_intensity"]:
    if col in df_train.columns:
        n = df_train[col].null_count()
        if n > 0:
            warn(f"  {col:<25} {n:>8,} Nulls")

print()
log("Starte Imputation auf Train...")
df_train_prep = impute_meteo_rolling(df_train)
log("Starte Imputation auf Test  (gleiche Funktion, keine neuen Parameter)...")
df_test_prep  = impute_meteo_rolling(df_test)

print()
log("Verbleibende Nulls nach Imputation (Train):")
remaining = 0
for col in METEO_COLS + ["flood_intensity"]:
    if col in df_train_prep.columns:
        n = df_train_prep[col].null_count()
        remaining += n
        if n > 0:
            warn(f"  {col:<25} {n:>8,} Nulls verbleiben (Rand-Stunden ohne Nachbarn)")
if remaining == 0:
    success("  Alle Meteo-Nulls gefüllt.")

# Export
df_train_prep.write_parquet(PROCESSED / "train_prepared.parquet")
df_test_prep.write_parquet(PROCESSED  / "test_prepared.parquet")

print()
success("Phase 3 abgeschlossen.")
log(f"  {PROCESSED / 'train_prepared.parquet'}  ({len(df_train_prep):,} Zeilen)")
log(f"  {PROCESSED / 'test_prepared.parquet'}   ({len(df_test_prep):,} Zeilen)")

## Phase 4 — Feature Engineering

Umsetzung der Feature-Ideen aus `01_exploration.ipynb` — Features Inspection.

| Kategorie | Feature | Basis | Begründung |
|:---|:---|:---|:---|
| **Zeit** | `stunde` | `arrival_schedule.dt.hour` | Tagesrhythmus |
| **Zeit** | `wochentag` | `arrival_schedule.dt.weekday` | Wochentag-Muster |
| **Zeit** | `monat` | `arrival_schedule.dt.month` | Saisonalität |
| **Zeit** | `ist_wochenende` | `wochentag >= 5` | Binäres Flag |
| **Zeit** | `ist_hvz` | `stunde in {7,8,9,17,18,19}` | Hauptverkehrszeit |
| **Wetter** | `hat_regen` | `precipitation > 0` | Zero-Inflation auflösen |
| **Wetter** | `hat_starkregen` | `precipitation > 5.0` | Schwellenwert-Effekt (R2) |
| **Wetter** | `hat_flut` | `flood_intensity > 0` | Stärkste Wetter-Korrelation r=0.15 |
| **Kategorial** | `line_name` enc. | `line_name` | Label- oder Target-Encoding |
| **Kategorial** | `district_name` enc. | `district_name` | inkl. `"ausserhalb"` |
| **Kategorial** | `event_size` enc. | `event_size` | Ordinal 0–3 |
| **Kategorial** | `is_canceled` | `canceled` | bool → int |

> Feature Engineering wird in diesem Notebook vorbereitet und in `03_modeling.ipynb` verfeinert.  
> Encoding-Parameter (Target-Encoding, Ordinal-Mappings) müssen auf `train_prepared` gefittet  
> und auf `test_prepared` angewendet werden — kein Leakage.

In [ ]:
section_header('Phase 4 — Feature Engineering (Zeitfeatures & Flags)')

# Zeitfeatures und binäre Flags — sicher vor/nach Split (keine Parameter gelernt)
def add_time_features(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns([
        pl.col("arrival_schedule").dt.hour().alias("stunde"),
        pl.col("arrival_schedule").dt.weekday().alias("wochentag"),
        pl.col("arrival_schedule").dt.month().alias("monat"),
        (pl.col("arrival_schedule").dt.weekday() >= 5).alias("ist_wochenende"),
        pl.col("arrival_schedule").dt.hour().is_in([7, 8, 9, 17, 18, 19]).alias("ist_hvz"),
    ])

def add_weather_flags(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns([
        (pl.col("precipitation") > 0).alias("hat_regen"),
        (pl.col("precipitation") > 5.0).alias("hat_starkregen"),
        (pl.col("flood_intensity") > 0).alias("hat_flut"),
        pl.col("canceled").cast(pl.Int8).alias("is_canceled"),
    ])

df_train_feat = df_train_prep.pipe(add_time_features).pipe(add_weather_flags)
df_test_feat  = df_test_prep.pipe(add_time_features).pipe(add_weather_flags)

# Encoding-Platzhalter — Umsetzung in 03_modeling.ipynb nach Feature Importance
# line_name / district_name: Label-Encoding oder Target-Encoding
# event_size: Ordinal 0–3

# Export
df_train_feat.write_parquet(PROCESSED / "train_features.parquet")
df_test_feat.write_parquet(PROCESSED  / "test_features.parquet")

log(f"Neue Features: {[c for c in df_train_feat.columns if c not in df_train_prep.columns]}")
print()
success("Phase 4 abgeschlossen.")
log(f"  {PROCESSED / 'train_features.parquet'}  ({len(df_train_feat):,} Zeilen)")
log(f"  {PROCESSED / 'test_features.parquet'}   ({len(df_test_feat):,} Zeilen)")